In [2]:
# 1 & 2. Import libraries and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
df = sns.load_dataset("titanic")

# 3. Inspect data
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

# 4. Handle missing values

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# Remove deck because it has many missing values
df = df.drop(columns=["deck"])

print("\nMissing Values:")
print(df.isnull().sum())

# 5. Remove duplicates

print("\nDuplicates:", df.duplicated().sum())

df = df.drop_duplicates()
df = df.reset_index(drop=True)

# 6. Standardize categorical values

df["sex"] = df["sex"].str.lower().str.strip()
df["embarked"] = df["embarked"].str.lower().str.strip()

# 9. Feature Engineering

df["FamilySize"] = df["sibsp"] + df["parch"] + 1

df["FamilyCategory"] = pd.cut(
    df["FamilySize"],
    bins=[0, 1, 4, np.inf],
    labels=["Alone", "Small", "Large"]
)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

# 6. One-Hot Encoding

categorical_columns = [
    "sex",
    "embarked",
    "class",
    "who",
    "alone",
    "FamilyCategory",
    "AgeGroup"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

# 7. Outlier Detection and Treatment

Q1 = df["fare"].quantile(0.25)
Q3 = df["fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("\nFare Lower Bound:", lower_bound)
print("Fare Upper Bound:", upper_bound)

# Cap outliers
df["fare"] = df["fare"].clip(
    lower=lower_bound,
    upper=upper_bound
)

# 8. Standardization

features_to_scale = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "FamilySize"
]

scaler = StandardScaler()

df[features_to_scale] = scaler.fit_transform(
    df[features_to_scale]
)

# 10. Save cleaned dataset

df.to_csv("titanic_cleaned.csv", index=False)

# Final result

print("\nFinal Dataset Shape:", df.shape)
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset:")
print(df.head())

print("\nCleaned dataset saved as titanic_cleaned.csv")

Dataset Shape: (891, 15)

First 5 rows:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ----